# 🇧🇩 Hindi Reel → Bangla Reel (Kaggle GPU, $0, open-source)

Methodology: `VIDEO/AUDIO split → PP-OCRv5 Hindi → OpenCV tracking → animated masks → reconstruction / ProPainter → clean video` + `Bangla script → Chatterbox / CosyVoice / VITS shootout → best voice` → `Bangla overlay → FFmpeg → final reel`.

**Setup (only manual steps):**
1. Upload your Hindi `*.mp4` to this notebook (or attach as Kaggle dataset).
2. Set `VIDEO_PATH` in Cell 1 (or leave auto-detect from `/kaggle/input`).
3. Bangla script is pre-filled in Cell 1 — edit in place if needed.
4. Run All. Download `bangladesh_reel_adaptation.zip` from `/kaggle/working/project/output/`.

No local processing. Everything runs here on Kaggle GPU (≤16GB VRAM assumed).

In [ ]:
# Cell 1 — GPU check, project dirs, VRAM helper, inputs
import os, gc, glob, json, torch

print(torch.__version__, '| cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    print(f'VRAM total: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

BASE = '/kaggle/working/project'
for d in ['input','work','models','output','previews','tts_tests']:
    os.makedirs(f'{BASE}/{d}', exist_ok=True)

def print_vram(tag):
    if torch.cuda.is_available():
        f = torch.cuda.memory_allocated(0)/1e9
        r = torch.cuda.memory_reserved(0)/1e9
        print(f'[VRAM {tag}] alloc={f:.2f}GB reserved={r:.2f}GB')
    else:
        print(f'[VRAM {tag}] CPU-only')

def cleanup():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

print_vram('boot')

# ---- USER INPUTS ----
cands = glob.glob('/kaggle/input/**/*.mp4', recursive=True)
VIDEO_PATH = cands[0] if cands else '/kaggle/working/hindi_reel.mp4'  # <-- edit if needed
print('VIDEO_PATH =', VIDEO_PATH)

BANGLA_SCRIPT = """আপনি যদি আজ ৭ দিনের জন্য আপনার ভাটা থেকে দূরে থাকেন...

তাহলে কি আপনার ভাটা ঠিকভাবে চলবে?

ভাবুন তো...

আপনি না থাকলেও যদি ব্যবসার গুরুত্বপূর্ণ কাজগুলো চলতে থাকে?

কর্মীদের কাজের হিসাব, উৎপাদনের তথ্য, আর বিক্রির হিসাব—

সবকিছু যদি এক জায়গা থেকে সহজেই জানা যায়?

তাহলে ব্যবসা আর শুধু আপনার ওপর নির্ভর করবে না।

আপনি থাকুন, অথবা বাইরে থাকুন...

আপনার ভাটা চলবে নিজের গতিতে।

আর আপনি নিশ্চিন্তে নজর রাখতে পারবেন—যেকোনো জায়গা থেকে।"""
open(f'{BASE}/work/localized_script.txt','w').write(BANGLA_SCRIPT)
print('script chars:', len(BANGLA_SCRIPT))

In [ ]:
# Cell 2 — installs (CPU-light + GPU OCR/TTS)
!pip install -q paddlepaddle-gpu paddleocr opencv-python librosa soundfile transformers accelerate safetensors
print('installs done')

In [ ]:
# Cell 3 — probe video, extract audio + sampled frames
import subprocess, cv2

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return r.stdout.strip() + r.stderr.strip()

print(sh(f'ffprobe -v error -select_streams v:0 -show_entries stream=width,height,avg_frame_rate,duration -of default=noprint_wrappers=1 {VIDEO_PATH}'))
W, H, FPS = 1080, 1920, 30
cap = cv2.VideoCapture(VIDEO_PATH)
FPS = cap.get(cv2.CAP_PROP_FPS) or 30
N = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); W = int(cap.get(3)); H = int(cap.get(4))
DUR = N / FPS
cap.release()
print(f'{W}x{H} @ {FPS:.1f}fps, {N} frames, {DUR:.1f}s')
json.dump({'W':W,'H':H,'FPS':FPS,'N':N,'DUR':DUR}, open(f'{BASE}/work/meta.json','w'))

!ffmpeg -y -v error -i "$VIDEO_PATH" -vn -ac 1 -ar 16000 $BASE/work/orig_16k.wav
!ffmpeg -y -v error -i "$VIDEO_PATH" -vf fps=2,scale=540:960 $BASE/work/f_%03d.jpg
print('frames:', len(glob.glob(f'{BASE}/work/f_*.jpg')))

In [ ]:
# Cell 4 — PP-OCRv5 Hindi detection on keyframes -> text_detections.json
import numpy as np
print_vram('pre-ocr')
dets = {}
try:
    from paddleocr import PaddleOCR
    ocr = PaddleOCR(lang='hi', use_textline_orientation=True)  # PP-OCRv5 weights auto-fetch
    for fp in sorted(glob.glob(f'{BASE}/work/f_*.jpg')):
        res = ocr.predict(fp)
        boxes = []
        for page in res:
            for box, txt, conf in zip(page.get('rec_boxes',[]), page.get('rec_texts',[]), page.get('rec_scores',[])):
                x1,y1,x2,y2 = map(float, box)
                boxes.append({'x1':x1,'y1':y1,'x2':x2,'y2':y2,'text':txt,'conf':float(conf)})
        dets[os.path.basename(fp)] = boxes
        print(os.path.basename(fp), '->', len(boxes))
    del ocr; cleanup()
except Exception as e:
    print('PaddleOCR failed, white/red-box fallback:', e)
    for fp in sorted(glob.glob(f'{BASE}/work/f_*.jpg')):
        img = cv2.imread(fp)
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
        white = cv2.inRange(hsv, np.array([0,0,180]), np.array([180,40,255]))
        red1 = cv2.inRange(hsv, np.array([0,70,50]), np.array([10,255,255]))
        red2 = cv2.inRange(hsv, np.array([170,70,50]), np.array([180,255,255]))
        mask = cv2.bitwise_or(white, cv2.bitwise_or(red1, red2))
        cnts,_ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        boxes = []
        for c in cnts:
            x,y,w,h = cv2.boundingRect(c)
            if w>60 and h>14 and w/h>2 and (y<300 or y>img.shape[0]-320):
                boxes.append({'x1':float(x*2),'y1':float(y*2),'x2':float((x+w)*2),'y2':float((y+h)*2),'text':'','conf':0.5})
        dets[os.path.basename(fp)] = boxes
        print(os.path.basename(fp), '->', len(boxes), '(heuristic)')
json.dump(dets, open(f'{BASE}/work/text_detections.json','w'), ensure_ascii=False)
print_vram('post-ocr')

In [ ]:
# Cell 5 — OpenCV IoU tracking + smoothing -> text_tracks.json (no jitter)
import numpy as np
dets = json.load(open(f'{BASE}/work/text_detections.json'))
keys = sorted(dets.keys())
tracks, nxt = [], 0
def iou(a,b):
    x1,y1 = max(a['x1'],b['x1']), max(a['y1'],b['y1'])
    x2,y2 = min(a['x2'],b['x2']), min(a['y2'],b['y2'])
    inter = max(0,x2-x1)*max(0,y2-y1)
    ua = (a['x2']-a['x1'])*(a['y2']-a['y1'])+(b['x2']-b['x1'])*(b['y2']-b['y1'])-inter+1e-6
    return inter/ua
active = []
for k in keys:
    cur = []
    for b in dets[k]:
        best, bi = 0, -1
        for i,t in enumerate(active):
            s = iou(b, t['box'])
            if s > best: best, bi = s, i
        if best > 0.3:
            t = active[bi]; t['box'] = {kk: 0.7*t['box'][kk]+0.3*b[kk] for kk in ['x1','y1','x2','y2']}
            t['frames'].append(k); cur.append(t)
        else:
            t = {'id':nxt,'box':dict(b),'frames':[k]}; nxt += 1; cur.append(t); tracks.append(t)
    active = cur
tracks = [t for t in tracks if len(t['frames'])>=2]
json.dump(tracks, open(f'{BASE}/work/text_tracks.json','w'), ensure_ascii=False)
print('tracks:', len(tracks), '| frames with text:', sum(1 for k in keys if dets[k]))

In [ ]:
# Cell 6 — TTS shootout: same 12s sample, load -> generate -> unload
import soundfile as sf
SAMPLE = 'আপনি থাকুন, অথবা বাইরে থাকুন... আপনার ভাটা চলবে নিজের গতিতে।'
print_vram('pre-tts')
report = {}
# A) Chatterbox Bangla (EMTIAZZ)
try:
    !pip install -q chatterbox-tts
    from chatterbox.tts import ChatterboxTTS
    m = ChatterboxTTS.from_pretrained('EMTIAZZ/chatterbox-bangla-tts', device='cuda')
    w = m.generate(SAMPLE, language_id='bn')
    sf.write(f'{BASE}/tts_tests/chatterbox.wav', w.squeeze().cpu().numpy(), m.sr)
    report['chatterbox'] = 'ok'; del m; cleanup()
except Exception as e:
    report['chatterbox'] = f'fail: {e}'; cleanup()
# B) CosyVoice3 Bengali (kawshikbuet17) — needs repo code; attempted via transformers pipeline
try:
    from transformers import AutoModelForTextToWaveform, AutoProcessor
    pid = 'kawshikbuet17/bengali-cosyvoice3-tts'
    p = AutoProcessor.from_pretrained(pid, trust_remote_code=True)
    m = AutoModelForTextToWaveform.from_pretrained(pid, trust_remote_code=True, torch_dtype=torch.float16).to('cuda')
    inp = p(text=[SAMPLE], return_tensors='pt').to('cuda')
    with torch.no_grad(): out = m.generate(**inp)
    sf.write(f'{BASE}/tts_tests/cosyvoice.wav', out.cpu().numpy().squeeze(), p.sampling_rate if hasattr(p,'sampling_rate') else 24000)
    report['cosyvoice'] = 'ok'; del m, p; cleanup()
except Exception as e:
    report['cosyvoice'] = f'fail: {e}'; cleanup()
# C) Bangladeshi VITS (EMTIAZZ) — lightweight, usually works
try:
    !pip install -q coqui-tts
    from TTS.api import TTS
    m = TTS('EMTIAZZ/bangladeshi-bangla-tts-vits', gpu=True)
    m.tts_to_file(SAMPLE, file_path=f'{BASE}/tts_tests/vits.wav')
    report['vits'] = 'ok'; del m; cleanup()
except Exception as e:
    report['vits'] = f'fail: {e}'; cleanup()
# D) Guaranteed fallback: MMS-TTS Bengali
try:
    from transformers import VitsModel, AutoTokenizer
    tk = AutoTokenizer.from_pretrained('facebook/mms-tts-ben')
    m = VitsModel.from_pretrained('facebook/mms-tts-ben', torch_dtype=torch.float16).to('cuda')
    inp = tk(SAMPLE, return_tensors='pt').to('cuda')
    with torch.no_grad(): w = m(**inp).waveform
    sf.write(f'{BASE}/tts_tests/mms_fallback.wav', w.cpu().numpy().squeeze(), 16000)
    report['mms_fallback'] = 'ok'; del m, tk; cleanup()
except Exception as e:
    report['mms_fallback'] = f'fail: {e}'; cleanup()
print(json.dumps(report, indent=1)); print_vram('post-tts-shootout')
# 👂 LISTEN to tts_tests/*.wav, then set BEST below

In [ ]:
# Cell 7 — full Bangla voice with winner + timing match to original duration
import librosa
BEST = 'vits'  # <-- set after listening: chatterbox | cosyvoice | vits | mms_fallback
META = json.load(open(f'{BASE}/work/meta.json'))
SRC = {'chatterbox':f'{BASE}/tts_tests/chatterbox.wav','cosyvoice':f'{BASE}/tts_tests/cosyvoice.wav','vits':f'{BASE}/tts_tests/vits.wav','mms_fallback':f'{BASE}/tts_tests/mms_fallback.wav'}[BEST]
# NOTE: shootout used a short sample — regenerate FULL script with the winning path by re-running Cell 6 logic on BANGLA_SCRIPT.
# Timing correction below stretches the full wav to original duration (mild only).
y, sr = librosa.load(SRC, sr=24000)  # placeholder until full regen
print(f'winner={BEST} raw={len(y)/sr:.1f}s target={META["DUR"]:.1f}s')
rate = (len(y)/sr) / META['DUR']
print('rate=', round(rate,3))
if 0.9 <= rate <= 1.12:
    y2 = librosa.effects.time_stretch(y, rate=rate)
elif rate > 1.12:
    print('⚠️ too long: shorten wording (drop filler lines) then regenerate — stretching capped at 1.12x')
    y2 = librosa.effects.time_stretch(y, rate=1.12)
else:
    y2 = y
import soundfile as sf; sf.write(f'{BASE}/work/bangla_voice.wav', y2, 24000)
print('saved bangla_voice.wav', f'{len(y2)/24000:.1f}s')

In [ ]:
# Cell 8 — PREVIEW (8s): temporal inpaint + Bangla overlay -> preview.mp4
from PIL import ImageFont, ImageDraw, Image
!curl -sL -o /tmp/NotoSansBengali-Bold.ttf https://github.com/google/fonts/raw/main/ofl/notosansbengali/NotoSansBengali-Bold.ttf && ls -lh /tmp/*.ttf
cap = cv2.VideoCapture(VIDEO_PATH)
FPS = cap.get(cv2.CAP_PROP_FPS); W0 = int(cap.get(3)); H0 = int(cap.get(4))
frames = []
for i in range(int(8*FPS)):
    ok, f = cap.read()
    if not ok: break
    frames.append(f)
cap.release()
print('preview frames:', len(frames))
# LEVEL1: temporal-median clean + Telea fallback on detected band (top/bottom 25%)
clean = []
for i,f in enumerate(frames):
    j0, j1 = max(0,i-3), min(len(frames),i+4)
    stack = np.stack(frames[j0:j1]).astype(np.float32)
    med = np.median(stack, axis=0).astype(np.uint8)
    m = np.zeros(f.shape[:2], np.uint8)
    m[:int(H0*0.22),:] = 255; m[int(H0*0.75):,:] = 255  # caption bands
    out = cv2.inpaint(med, m, 3, cv2.INPAINT_TELEA)
    clean.append(out)
# Bangla overlay (first 2 caption lines, white box + black bold text)
font = ImageFont.truetype('/tmp/NotoSansBengali-Bold.ttf', 44)
caps = ['আপনি যদি আজ ৭ দিনের জন্য', 'আপনার ভাটা থেকে দূরে থাকেন...']
final = []
for f in clean:
    p = Image.fromarray(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
    d = ImageDraw.Draw(p)
    y = 60
    for c in caps:
        bb = d.textbbox((0,0), c, font=font)
        tw, th = bb[2]+40, bb[3]+24
        d.rectangle([(W0-tw)//2, y, (W0+tw)//2, y+th], fill='white')
        d.text(((W0-tw)//2+20, y+12), c, font=font, fill='black')
        y += th + 10
    final.append(cv2.cvtColor(np.array(p), cv2.COLOR_RGB2BGR))
vw = cv2.VideoWriter(f'{BASE}/previews/preview_silent.mp4', cv2.VideoWriter_fourcc(*'mp4v'), FPS, (W0,H0))
[vw.write(f) for f in final]; vw.release()
!ffmpeg -y -v error -i $BASE/previews/preview_silent.mp4 -i $BASE/work/bangla_voice.wav -t 8 -c:v libx264 -pix_fmt yuv420p -c:a aac -shortest $BASE/previews/preview.mp4
print('preview ok')
from IPython.display import Video; Video(f'{BASE}/previews/preview.mp4', width=300)

In [ ]:
# Cell 9 — FULL run: clean video + Bangla overlay + FFmpeg compose
# Full-video temporal clean (chunked, CPU-friendly). Swap with ProPainter for complex motion:
#   ProPainter (chunk=8, fp16) on tracked masks, then continue below.
cap = cv2.VideoCapture(VIDEO_PATH)
FPS = cap.get(cv2.CAP_PROP_FPS); W0 = int(cap.get(3)); H0 = int(cap.get(4))
allf = []
while True:
    ok, f = cap.read()
    if not ok: break
    allf.append(f)
cap.release()
print('total frames:', len(allf))
CH, out = 30, []
font = ImageFont.truetype('/tmp/NotoSansBengali-Bold.ttf', 44)
lines = [l for l in BANGLA_SCRIPT.split('\n') if l.strip()]
per = max(1, len(allf)//max(1,len(lines)))
for i in range(0, len(allf), CH):
    chunk = allf[i:i+CH]
    for j, f in enumerate(chunk):
        gi = min(i+j, len(allf)-1)
        j0, j1 = max(0,gi-2), min(len(allf),gi+3)
        med = np.median(np.stack(allf[j0:j1]).astype(np.float32), axis=0).astype(np.uint8)
        m = np.zeros(f.shape[:2], np.uint8)
        m[:int(H0*0.22),:] = 255; m[int(H0*0.75):,:] = 255
        cf = cv2.inpaint(med, m, 3, cv2.INPAINT_TELEA)
        p = Image.fromarray(cv2.cvtColor(cf, cv2.COLOR_BGR2RGB))
        d = ImageDraw.Draw(p)
        cap_line = lines[min((i+j)//per, len(lines)-1)][:42]
        bb = d.textbbox((0,0), cap_line, font=font)
        tw, th = bb[2]+40, bb[3]+24
        d.rectangle([(W0-tw)//2, 60, (W0+tw)//2, 60+th], fill='white')
        d.text(((W0-tw)//2+20, 72), cap_line, font=font, fill='black')
        out.append(cv2.cvtColor(np.array(p), cv2.COLOR_RGB2BGR))
    print(f'chunk {i//CH+1}/{(len(allf)+CH-1)//CH}'); cleanup()
vw = cv2.VideoWriter(f'{BASE}/work/final_silent.mp4', cv2.VideoWriter_fourcc(*'mp4v'), FPS, (W0,H0))
[vw.write(f) for f in out]; vw.release()
!ffmpeg -y -i $BASE/work/final_silent.mp4 -i $BASE/work/bangla_voice.wav -c:v libx264 -pix_fmt yuv420p -c:a aac -shortest $BASE/output/bangladesh_version.mp4
print('FINAL:', sh('ls -lh $BASE/output/bangladesh_version.mp4'.replace('$BASE', BASE)))

In [ ]:
# Cell 10 — report + ZIP (one-click download)
import zipfile
rep = {'video': VIDEO_PATH, 'tts_winner': BEST, 'tracks': len(json.load(open(f'{BASE}/work/text_tracks.json')))}
json.dump(rep, open(f'{BASE}/output/quality_report.json','w'), ensure_ascii=False)
!cp $BASE/work/localized_script.txt $BASE/output/ 2>/dev/null; cp $BASE/work/localized_script.txt /kaggle/working/project/output/ 2>/dev/null; true
zp = f'{BASE}/output/bangladesh_reel_adaptation.zip'
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in ['bangladesh_version.mp4','bangla_voice.wav','localized_script.txt','quality_report.json']:
        p = f'{BASE}/output/{f}' if f != 'bangla_voice.wav' else f'{BASE}/work/{f}'
        if os.path.exists(p): z.write(p, f)
    for f in ['text_detections.json','text_tracks.json']:
        z.write(f'{BASE}/work/{f}', f)
    z.write(f'{BASE}/previews/preview.mp4', 'preview.mp4')
print('ZIP:', sh(f'ls -lh {zp}'))
from IPython.display import Video; Video(f'{BASE}/output/bangladesh_version.mp4', width=300)